In [1]:
# ============================================================
# Q1: Polyalphabetic Cryptanalysis via Kasiski Examination
# ============================================================

from math import gcd
from functools import reduce
from collections import Counter, defaultdict

# --- Hex-decode the ciphertext ---
hex_ct = ("444d47434d51534b46574f53525147444b4c484f47554853"
          "444254474f4b4f424b514d42514343414953574d")
ciphertext = bytes.fromhex(hex_ct).decode('latin-1').upper()

print("=" * 60)
print("Q1: KASISKI EXAMINATION")
print("=" * 60)
print(f"Decoded Ciphertext: {ciphertext}")
print(f"Length: {len(ciphertext)} characters\n")

# --- Step 1: Find all repeated n-grams (n >= 3) ---
print("Step 1: Searching for repeated n-grams (n >= 3)...")
repeats = {}
for n in range(3, len(ciphertext) // 2 + 1):
    for i in range(len(ciphertext) - n + 1):
        ngram = ciphertext[i:i+n]
        positions = [j for j in range(i+1, len(ciphertext)-n+1)
                     if ciphertext[j:j+n] == ngram]
        if positions:
            if ngram not in repeats:
                repeats[ngram] = [i] + positions

distances = []
if repeats:
    for ngram, positions in sorted(repeats.items(), key=lambda x: -len(x[0])):
        for k in range(len(positions)-1):
            d = positions[k+1] - positions[k]
            distances.append(d)
            print(f"  Repeated n-gram: '{ngram}' at positions "
                  f"{positions[k]} and {positions[k+1]}, distance = {d}")
else:
    print("  No repeated n-grams found (ciphertext is short).")
    print("  Falling back to Index of Coincidence (IoC) method.\n")

# --- Step 2: GCD of distances ---
print("\nStep 2: Computing GCD of all distances...")
if distances:
    overall_gcd = reduce(gcd, distances)
    print(f"  GCD of distances {distances} = {overall_gcd}")
else:
    print("  No distances to compute GCD from.")

# --- Step 3: IoC for each key length ---
print("\nStep 3: Index of Coincidence (IoC) for key length confirmation...")
print("  (English IoC ≈ 0.065, Random ≈ 0.038)")

def index_of_coincidence(text):
    text = [c for c in text if c.isalpha()]
    n = len(text)
    if n < 2:
        return 0
    freq = Counter(text)
    return sum(f*(f-1) for f in freq.values()) / (n*(n-1))

ioc_scores = {}
for kl in range(2, 10):
    groups = [''.join(ciphertext[i] for i in range(j, len(ciphertext), kl))
              for j in range(kl)]
    avg_ioc = sum(index_of_coincidence(g) for g in groups) / kl
    ioc_scores[kl] = avg_ioc
    print(f"  Key length {kl}: avg IoC = {avg_ioc:.4f}")

top3 = sorted(ioc_scores, key=ioc_scores.get, reverse=True)[:3]
print(f"\n  Top 3 probable key lengths: {top3}")

# --- Step 4: Recover keyword via frequency analysis ---
ENGLISH_FREQ = {
    'E':12.7,'T':9.1,'A':8.2,'O':7.5,'I':7.0,'N':6.7,'S':6.3,
    'H':6.1,'R':6.0,'D':4.3,'L':4.0,'C':2.8,'U':2.8,'M':2.4,
    'W':2.4,'F':2.2,'G':2.0,'Y':2.0,'P':1.9,'B':1.5,'V':1.0,
    'K':0.8,'J':0.2,'X':0.2,'Q':0.1,'Z':0.1
}

def best_shift_for_group(group):
    best_sh, best_score = 0, -999
    for shift in range(26):
        dec = ''.join(chr((ord(c)-ord('A')-shift) % 26 + ord('A'))
                      for c in group if c.isalpha())
        freq = Counter(dec)
        score = sum(freq.get(ch, 0) * ENGLISH_FREQ.get(ch, 0)
                    for ch in 'ABCDEFGHIJKLMNOPQRSTUVWXYZ')
        if score > best_score:
            best_score, best_sh = score, shift
    return best_sh, chr(best_sh + ord('A'))

print("\nStep 4: Recovering keyword for each top key length...")
for kl in top3:
    groups = [''.join(ciphertext[i] for i in range(j, len(ciphertext), kl))
              for j in range(kl)]
    key = ''.join(best_shift_for_group(g)[1] for g in groups)
    ki = 0
    decrypted = []
    for c in ciphertext:
        if c.isalpha():
            shift = ord(key[ki % len(key)]) - ord('A')
            decrypted.append(chr((ord(c) - ord('A') - shift) % 26 + ord('A')))
            ki += 1
        else:
            decrypted.append(c)
    print(f"\n  Key length {kl}: Keyword = '{key}'")
    print(f"  Decrypted text: {''.join(decrypted)}")

Q1: KASISKI EXAMINATION
Decoded Ciphertext: DMGCMQSKFWOSRQGDKLHOGUHSDBTGOKOBKQMBQCCAISWM
Length: 44 characters

Step 1: Searching for repeated n-grams (n >= 3)...
  No repeated n-grams found (ciphertext is short).
  Falling back to Index of Coincidence (IoC) method.


Step 2: Computing GCD of all distances...
  No distances to compute GCD from.

Step 3: Index of Coincidence (IoC) for key length confirmation...
  (English IoC ≈ 0.065, Random ≈ 0.038)
  Key length 2: avg IoC = 0.0476
  Key length 3: avg IoC = 0.0764
  Key length 4: avg IoC = 0.0455
  Key length 5: avg IoC = 0.0627
  Key length 6: avg IoC = 0.0813
  Key length 7: avg IoC = 0.0449
  Key length 8: avg IoC = 0.0292
  Key length 9: avg IoC = 0.0889

  Top 3 probable key lengths: [9, 6, 3]

Step 4: Recovering keyword for each top key length...

  Key length 9: Keyword = 'DKCNIOZIX'
  Decrypted text: ACEPECTCITEQEISECOEEEHZEETWDEIBTWREENSANAEXE

  Key length 6: Keyword = 'DICCKO'
  Decrypted text: AEEACCPCDUEEOIEBAXEGESXEATREEW

In [2]:
# ============================================================
# Q2: DES f-Function Trace
# ============================================================

R0 = 0x01234567
K1 = 0x133457799BBC

print("=" * 60)
print("Q2: DES f-FUNCTION TRACE")
print("=" * 60)
print(f"Input  R0 = {hex(R0).upper()} (32-bit)")
print(f"Subkey K1 = {hex(K1).upper()} (48-bit)\n")

# DES Expansion Table
E_TABLE = [
    32, 1, 2, 3, 4, 5,
     4, 5, 6, 7, 8, 9,
     8, 9,10,11,12,13,
    12,13,14,15,16,17,
    16,17,18,19,20,21,
    20,21,22,23,24,25,
    24,25,26,27,28,29,
    28,29,30,31,32, 1
]

# All 8 DES S-Boxes
S_BOXES = [
    [[14,4,13,1,2,15,11,8,3,10,6,12,5,9,0,7],
     [0,15,7,4,14,2,13,1,10,6,12,11,9,5,3,8],
     [4,1,14,8,13,6,2,11,15,12,9,7,3,10,5,0],
     [15,12,8,2,4,9,1,7,5,11,3,14,10,0,6,13]],
    [[15,1,8,14,6,11,3,4,9,7,2,13,12,0,5,10],
     [3,13,4,7,15,2,8,14,12,0,1,10,6,9,11,5],
     [0,14,7,11,10,4,13,1,5,8,12,6,9,3,2,15],
     [13,8,10,1,3,15,4,2,11,6,7,12,0,5,14,9]],
    [[10,0,9,14,6,3,15,5,1,13,12,7,11,4,2,8],
     [13,7,0,9,3,4,6,10,2,8,5,14,12,11,15,1],
     [13,6,4,9,8,15,3,0,11,1,2,12,5,10,14,7],
     [1,10,13,0,6,9,8,7,4,15,14,3,11,5,2,12]],
    [[7,13,14,3,0,6,9,10,1,2,8,5,11,12,4,15],
     [13,8,11,5,6,15,0,3,4,7,2,12,1,10,14,9],
     [10,6,9,0,12,11,7,13,15,1,3,14,5,2,8,4],
     [3,15,0,6,10,1,13,8,9,4,5,11,12,7,2,14]],
    [[2,12,4,1,7,10,11,6,8,5,3,15,13,0,14,9],
     [14,11,2,12,4,7,13,1,5,0,15,10,3,9,8,6],
     [4,2,1,11,10,13,7,8,15,9,12,5,6,3,0,14],
     [11,8,12,7,1,14,2,13,6,15,0,9,10,4,5,3]],
    [[12,1,10,15,9,2,6,8,0,13,3,4,14,7,5,11],
     [10,15,4,2,7,12,9,5,6,1,13,14,0,11,3,8],
     [9,14,15,5,2,8,12,3,7,0,4,10,1,13,11,6],
     [4,3,2,12,9,5,15,10,11,14,1,7,6,0,8,13]],
    [[4,11,2,14,15,0,8,13,3,12,9,7,5,10,6,1],
     [13,0,11,7,4,9,1,10,14,3,5,12,2,15,8,6],
     [1,4,11,13,12,3,7,14,10,15,6,8,0,5,9,2],
     [6,11,13,8,1,4,10,7,9,5,0,15,14,2,3,12]],
    [[13,2,8,4,6,15,11,1,10,9,3,14,5,0,12,7],
     [1,15,13,8,10,3,7,4,12,5,6,11,0,14,9,2],
     [7,11,4,1,9,12,14,2,0,6,10,13,15,3,5,8],
     [2,1,14,7,4,10,8,13,15,12,9,0,3,5,6,11]]
]

# DES P-box
P_TABLE = [
    16,7,20,21,29,12,28,17,
     1,15,23,26, 5,18,31,10,
     2, 8,24,14,32,27, 3, 9,
    19,13,30, 6,22,11, 4,25
]

# Step 1: Expansion
R0_bits = format(R0, '032b')
expanded_bits = ''.join(R0_bits[E_TABLE[i]-1] for i in range(48))
print(f"Step 1 — Expansion E(R0):")
print(f"  R0 in binary:       {R0_bits}")
print(f"  Expanded (48-bit):  {expanded_bits}")
print(f"  Expanded hex:       {hex(int(expanded_bits, 2)).upper()}\n")

# Step 2: XOR with K1
K1_bits = format(K1, '048b')
xored_bits = ''.join(str(int(a) ^ int(b)) for a, b in zip(expanded_bits, K1_bits))
print(f"Step 2 — XOR with K1:")
print(f"  K1 binary:          {K1_bits}")
print(f"  XOR result (48-bit):{xored_bits}")
print(f"  XOR result hex:     {hex(int(xored_bits, 2)).upper()}\n")

# Step 3: S-Box substitution
print(f"Step 3 — S-Box Substitution:")
print(f"  (Row index = MSB + LSB of 6-bit chunk, Column = middle 4 bits)\n")
sbox_output_bits = ''
for i in range(8):
    chunk = xored_bits[i*6:(i+1)*6]
    row = int(chunk[0] + chunk[5], 2)
    col = int(chunk[1:5], 2)
    val = S_BOXES[i][row][col]
    val_bits = format(val, '04b')
    sbox_output_bits += val_bits
    if i < 2:
        print(f"  S-Box {i+1}: input = {chunk} | MSB={chunk[0]}, LSB={chunk[5]} "
              f"→ row={row} | middle bits={chunk[1:5]} → col={col} "
              f"| output = {val} = {val_bits}  ← DETAILED TRACE")
    else:
        print(f"  S-Box {i+1}: input={chunk}, row={row}, col={col}, "
              f"output={val} ({val_bits})")

print(f"\n  S-Box combined output (32-bit): {sbox_output_bits}")
print(f"  S-Box output hex: {hex(int(sbox_output_bits, 2)).upper()}\n")

# Step 4: P-box permutation
p_output_bits = ''.join(sbox_output_bits[P_TABLE[i]-1] for i in range(32))
print(f"Step 4 — Final P-box Permutation:")
print(f"  Input:  {sbox_output_bits}")
print(f"  Output: {p_output_bits}")
print(f"\n  ✅ f(R0, K1) = {hex(int(p_output_bits, 2)).upper()}")

Q2: DES f-FUNCTION TRACE
Input  R0 = 0X1234567 (32-bit)
Subkey K1 = 0X133457799BBC (48-bit)

Step 1 — Expansion E(R0):
  R0 in binary:       00000001001000110100010101100111
  Expanded (48-bit):  100000000010100100000110101000001010101100001110
  Expanded hex:       0X802906A0AB0E

Step 2 — XOR with K1:
  K1 binary:          000100110011010001010111011110011001101110111100
  XOR result (48-bit):100100110001110101010001110110010011000010110010
  XOR result hex:     0X931D51D930B2

Step 3 — S-Box Substitution:
  (Row index = MSB + LSB of 6-bit chunk, Column = middle 4 bits)

  S-Box 1: input = 100100 | MSB=1, LSB=0 → row=2 | middle bits=0010 → col=2 | output = 14 = 1110  ← DETAILED TRACE
  S-Box 2: input = 110001 | MSB=1, LSB=1 → row=3 | middle bits=1000 → col=8 | output = 11 = 1011  ← DETAILED TRACE
  S-Box 3: input=110101, row=3, col=10, output=14 (1110)
  S-Box 4: input=010001, row=1, col=8, output=4 (0100)
  S-Box 5: input=110110, row=2, col=11, output=5 (0101)
  S-Box 6: input=01001

In [ ]:
# ============================================================
# Q3: AES State Matrix Manipulation
# ============================================================

import copy

# NIST standard AES S-Box (16x16 lookup table)
AES_SBOX = [
    [0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76],
    [0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0],
    [0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15],
    [0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75],
    [0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84],
    [0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf],
    [0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8],
    [0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2],
    [0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73],
    [0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb],
    [0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79],
    [0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08],
    [0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a],
    [0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e],
    [0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf],
    [0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16]
]

def print_matrix(matrix, title):
    print(f"\n  {title}:")
    for row in matrix:
        print("  " + " ".join(f"{b:02X}" for b in row))

print("=" * 60)
print("Q3: AES STATE MATRIX MANIPULATION")
print("=" * 60)

hex_input = "3243F6A8885A308D313198A2E0370734"
raw_bytes  = bytes.fromhex(hex_input)
print(f"Input hex string: {hex_input}\n")

# Build state matrix — filled column by column (AES spec)
state = [[0]*4 for _ in range(4)]
for col in range(4):
    for row in range(4):
        state[row][col] = raw_bytes[col*4 + row]

print_matrix(state, "Initial State Matrix (filled column by column)")
print("  Note: bytes fill column 0 first (top to bottom), then column 1, etc.")

# Step 1: SubBytes show S-BOX before ShiftRows
subbytes = copy.deepcopy(state)
print("\n  SubBytes lookup detail (first 4 bytes shown):")
for r in range(4):
    for c in range(4):
        byte = state[r][c]
        row_idx = (byte >> 4) & 0xF    # upper 4 bits = row index
        col_idx = byte & 0xF           # lower 4 bits = col index
        subbytes[r][c] = AES_SBOX[row_idx][col_idx]
        if r == 0:
            print(f"    byte {byte:02X} → row={row_idx}, col={col_idx} "
                  f"→ S-Box[{row_idx}][{col_idx}] = {subbytes[r][c]:02X}")

print_matrix(subbytes, "After SubBytes (NIST AES S-Box — applied first)")

# Step 2: ShiftRows SECOND
shifted = copy.deepcopy(subbytes)
for r in range(1, 4):
    shifted[r] = subbytes[r][r:] + subbytes[r][:r]

print_matrix(shifted, "After ShiftRows (applied second)")
print("  Row 0: shift left by 0 (unchanged)")
print("  Row 1: shift left by 1")
print("  Row 2: shift left by 2")
print("  Row 3: shift left by 3")

print("\n  ✅ Final transformed state matrix printed above.")

Q3: AES STATE MATRIX MANIPULATION
Input hex string: 3243F6A8885A308D313198A2E0370734


  Initial State Matrix (filled column by column):
  32 88 31 E0
  43 5A 31 37
  F6 30 98 07
  A8 8D A2 34
  Note: bytes fill column 0 first (top to bottom), then column 1, etc.

  SubBytes lookup detail (first 4 bytes shown):
    byte 32 → row=3, col=2 → S-Box[3][2] = 23
    byte 88 → row=8, col=8 → S-Box[8][8] = C4
    byte 31 → row=3, col=1 → S-Box[3][1] = C7
    byte E0 → row=14, col=0 → S-Box[14][0] = E1

  After SubBytes (NIST AES S-Box — applied first):
  23 C4 C7 E1
  1A BE C7 9A
  42 04 46 C5
  C2 5D 3A 18

  After ShiftRows (applied second):
  23 C4 C7 E1
  BE C7 9A 1A
  46 C5 42 04
  18 C2 5D 3A
  Row 0: shift left by 0 (unchanged)
  Row 1: shift left by 1
  Row 2: shift left by 2
  Row 3: shift left by 3

  ✅ Final transformed state matrix printed above.


In [4]:
# ============================================================
# Q4: Visual Cryptanalysis — ECB vs CBC
# Run in Google Colab
# ============================================================

!pip install cryptography

import os, struct
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend

print("=" * 60)
print("Q4: ECB vs CBC VISUAL CRYPTANALYSIS")
print("=" * 60)

def aes_pad(data, block_size=16):
    pad_len = block_size - (len(data) % block_size)
    return data + bytes([pad_len] * pad_len)

# --- Create a synthetic 128x128 striped bitmap ---
# (Replace with a real image file if you have one)
print("Creating striped bitmap image (tux.bmp)...")
width, height = 128, 128
row_size = width * 3
bmp_padding = (4 - row_size % 4) % 4
padded_row = row_size + bmp_padding
file_size = 54 + padded_row * height

with open("tux.bmp", "wb") as f:
    # BMP File Header
    f.write(b'BM')
    f.write(struct.pack('<I', file_size))
    f.write(struct.pack('<HH', 0, 0))
    f.write(struct.pack('<I', 54))
    # DIB Header
    f.write(struct.pack('<I', 40))
    f.write(struct.pack('<i', width))
    f.write(struct.pack('<i', -height))
    f.write(struct.pack('<HH', 1, 24))
    f.write(struct.pack('<I', 0))
    f.write(struct.pack('<I', padded_row * height))
    f.write(struct.pack('<i', 2835))
    f.write(struct.pack('<i', 2835))
    f.write(struct.pack('<I', 0))
    f.write(struct.pack('<I', 0))
    # Pixel data — bold coloured stripes every 16 rows
    colors = [(255,0,0),(0,255,0),(0,0,255),(255,255,0),(0,255,255),(255,0,255)]
    for row in range(height):
        color = colors[(row // 16) % len(colors)]
        for col in range(width):
            f.write(bytes([color[2], color[1], color[0]]))  # BGR order
        f.write(b'\x00' * bmp_padding)

print("tux.bmp created.\n")

# --- Read bitmap ---
with open("tux.bmp", "rb") as f:
    bmp_data = f.read()

header     = bmp_data[:54]   # preserve BMP header unchanged
pixel_data = bmp_data[54:]
key        = b'\x00' * 16    # AES-128 key
padded_pixels = aes_pad(pixel_data)

# --- ECB Encryption ---
print("Encrypting with ECB...")
cipher_ecb = Cipher(algorithms.AES(key), modes.ECB(), backend=default_backend())
enc = cipher_ecb.encryptor()
ecb_encrypted = (enc.update(padded_pixels) + enc.finalize())[:len(pixel_data)]
with open("ecb_encrypted.bmp", "wb") as f:
    f.write(header + ecb_encrypted)
print("Saved: ecb_encrypted.bmp")

# --- CBC Encryption ---
print("Encrypting with CBC...")
iv = os.urandom(16)
cipher_cbc = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
enc = cipher_cbc.encryptor()
cbc_encrypted = (enc.update(padded_pixels) + enc.finalize())[:len(pixel_data)]
with open("cbc_encrypted.bmp", "wb") as f:
    f.write(header + cbc_encrypted)
print("Saved: cbc_encrypted.bmp\n")

print("ECB Formula:  Cᵢ = Eₖ(Pᵢ)")
print("  → Identical plaintext blocks produce IDENTICAL ciphertext blocks.")
print("  → Repeating pixel regions stay visually recognisable after encryption.")
print("  → Semantic security FAILS.\n")
print("CBC Formula:  Cᵢ = Eₖ(Pᵢ ⊕ Cᵢ₋₁),  C₀ = IV")
print("  → Every block depends on all previous ciphertext.")
print("  → Even identical plaintext blocks produce different ciphertext.")
print("  → Output looks like random noise. Semantic security HOLDS.\n")

# Display images inline in Colab
try:
    from IPython.display import Image, display
    print("Original Image:")
    display(Image("tux.bmp"))
    print("ECB Encrypted (structure visible):")
    display(Image("ecb_encrypted.bmp"))
    print("CBC Encrypted (random noise):")
    display(Image("cbc_encrypted.bmp"))
except:
    pass

print("\n[Download ecb_encrypted.bmp and cbc_encrypted.bmp from the Colab file browser]")

Q4: ECB vs CBC VISUAL CRYPTANALYSIS
Creating striped bitmap image (tux.bmp)...
tux.bmp created.

Encrypting with ECB...
Saved: ecb_encrypted.bmp
Encrypting with CBC...
Saved: cbc_encrypted.bmp

ECB Formula:  Cᵢ = Eₖ(Pᵢ)
  → Identical plaintext blocks produce IDENTICAL ciphertext blocks.
  → Repeating pixel regions stay visually recognisable after encryption.
  → Semantic security FAILS.

CBC Formula:  Cᵢ = Eₖ(Pᵢ ⊕ Cᵢ₋₁),  C₀ = IV
  → Every block depends on all previous ciphertext.
  → Even identical plaintext blocks produce different ciphertext.
  → Output looks like random noise. Semantic security HOLDS.

Original Image:

[Download ecb_encrypted.bmp and cbc_encrypted.bmp from the Colab file browser]


In [5]:
# ============================================================
# Q5: Resource-Constrained MAC — CMAC Implementation
# Run in Google Colab
# ============================================================

!pip install cryptography

from cryptography.hazmat.primitives.cmac import CMAC
from cryptography.hazmat.primitives.ciphers import algorithms
from cryptography.hazmat.backends import default_backend

print("=" * 60)
print("Q5: CMAC IMPLEMENTATION")
print("=" * 60)

key = b'CSCI6000 KEY1234'   # 128-bit AES key (16 bytes)
msg = b'UPDATE FIRMWARE'

print(f"Key (128-bit):  {key.decode()}")
print(f"Message:        {msg.decode()}\n")

# Generate CMAC tag using AES
c = CMAC(algorithms.AES(key), backend=default_backend())
c.update(msg)
tag = c.finalize()

print(f"CMAC Tag (hex): {tag.hex().upper()}")
print(f"Tag length:     {len(tag) * 8} bits ({len(tag)} bytes)\n")

print("Why CMAC and NOT HMAC?")
print("  HMAC requires a cryptographic hash function (e.g., SHA-256 or SHA-1).")
print("  This embedded IoT device has a hardware AES accelerator but NO hash engine.")
print("  CMAC is defined in NIST SP 800-38B and uses only AES block cipher operations.")
print("  Since AES is available in hardware, CMAC can be computed on this device.")
print("  HMAC is physically impossible to run here — CMAC is the correct solution.")

Q5: CMAC IMPLEMENTATION
Key (128-bit):  CSCI6000 KEY1234
Message:        UPDATE FIRMWARE

CMAC Tag (hex): 9800C6750448697627BFE7DC99950197
Tag length:     128 bits (16 bytes)

Why CMAC and NOT HMAC?
  HMAC requires a cryptographic hash function (e.g., SHA-256 or SHA-1).
  This embedded IoT device has a hardware AES accelerator but NO hash engine.
  CMAC is defined in NIST SP 800-38B and uses only AES block cipher operations.
  Since AES is available in hardware, CMAC can be computed on this device.
  HMAC is physically impossible to run here — CMAC is the correct solution.


In [6]:
# ============================================================
# Q6: RSA Decryption via Extended Euclidean Algorithm
# ============================================================

print("=" * 60)
print("Q6: RSA DECRYPTION — EXTENDED EUCLIDEAN ALGORITHM")
print("=" * 60)

C = 894    # Intercepted ciphertext
n = 1079   # Public modulus
e = 43     # Public exponent

print(f"Ciphertext C = {C}")
print(f"Public key:  n = {n},  e = {e}\n")

# --- Step 1: Factorize n ---
print("Step 1: Factorizing n into primes p and q...")
p, q = None, None
for i in range(2, int(n**0.5) + 1):
    if n % i == 0:
        p, q = i, n // i
        break
print(f"  {n} = {p} × {q}  ✓")

# --- Step 2: Euler's Totient ---
phi_n = (p - 1) * (q - 1)
print(f"\nStep 2: Compute φ(n) = (p−1)(q−1)")
print(f"  φ({n}) = ({p}−1) × ({q}−1) = {p-1} × {q-1} = {phi_n}")

# --- Step 3: Extended Euclidean Algorithm ---
print(f"\nStep 3: Extended Euclidean Algorithm")
print(f"  Find d such that: d × {e} ≡ 1 (mod {phi_n})\n")

def extended_gcd(a, b, depth=0):
    indent = "  " + "    " * depth
    if b == 0:
        print(f"{indent}ext_gcd({a}, 0) → base case: gcd=1, x=1, y=0")
        return a, 1, 0
    print(f"{indent}ext_gcd({a}, {b}) →")
    g, x, y = extended_gcd(b, a % b, depth + 1)
    new_x = y
    new_y = x - (a // b) * y
    print(f"{indent}  ← returns gcd={g}, x={new_x}, y={new_y}")
    return g, new_x, new_y

g, x, _ = extended_gcd(e, phi_n)
d = x % phi_n

print(f"\n  Private exponent d = {x} mod {phi_n} = {d}")
print(f"  Verification: ({d} × {e}) mod {phi_n} = {(d*e) % phi_n}  ✓")

# --- Step 4: Decrypt ---
M = pow(C, d, n)
print(f"\nStep 4: Decrypt")
print(f"  M = C^d mod n")
print(f"  M = {C}^{d} mod {n}")
print(f"\n  ✅ Recovered Plaintext M = {M}")

Q6: RSA DECRYPTION — EXTENDED EUCLIDEAN ALGORITHM
Ciphertext C = 894
Public key:  n = 1079,  e = 43

Step 1: Factorizing n into primes p and q...
  1079 = 13 × 83  ✓

Step 2: Compute φ(n) = (p−1)(q−1)
  φ(1079) = (13−1) × (83−1) = 12 × 82 = 984

Step 3: Extended Euclidean Algorithm
  Find d such that: d × 43 ≡ 1 (mod 984)

  ext_gcd(43, 984) →
      ext_gcd(984, 43) →
          ext_gcd(43, 38) →
              ext_gcd(38, 5) →
                  ext_gcd(5, 3) →
                      ext_gcd(3, 2) →
                          ext_gcd(2, 1) →
                              ext_gcd(1, 0) → base case: gcd=1, x=1, y=0
                            ← returns gcd=1, x=0, y=1
                        ← returns gcd=1, x=1, y=-1
                    ← returns gcd=1, x=-1, y=2
                ← returns gcd=1, x=2, y=-15
            ← returns gcd=1, x=-15, y=17
        ← returns gcd=1, x=17, y=-389
    ← returns gcd=1, x=-389, y=17

  Private exponent d = -389 mod 984 = 595
  Verification: (595 × 43) mod 

In [7]:
# ============================================================
# Q7: Primitive Roots & Diffie-Hellman Keyspace
# ============================================================

from math import gcd

print("=" * 60)
print("Q7: PRIMITIVE ROOTS FOR p = 23")
print("=" * 60)

p = 23

print(f"Prime p = {p}")
print(f"Definition: g is a primitive root of p if the set")
print(f"  {{g^1 mod p, g^2 mod p, ..., g^(p-1) mod p}}")
print(f"  generates every integer in {{1, 2, ..., {p-1}}} exactly once.\n")

# Find all primitive roots
primitive_roots = []
print("Testing each candidate g from 2 to p-1:\n")
for g in range(2, p):
    powers = set(pow(g, k, p) for k in range(1, p))
    expected = set(range(1, p))
    is_root = (powers == expected)
    if is_root:
        primitive_roots.append(g)
        print(f"  g = {g:2d}  → powers = {sorted(powers)}  ✓ PRIMITIVE ROOT")
    else:
        missing = expected - powers
        print(f"  g = {g:2d}  → missing {sorted(missing)[:3]}{'...' if len(missing)>3 else ''}  ✗")

print(f"\n✅ All primitive roots mod {p}: {primitive_roots}")
print(f"   Total count: {len(primitive_roots)}\n")

# Euler's Totient relationship
phi_p   = p - 1
phi_phi = sum(1 for k in range(1, phi_p + 1) if gcd(k, phi_p) == 1)

print(f"Mathematical relationship:")
print(f"  φ(p)    = φ({p})   = {phi_p}  (p is prime, so φ(p) = p−1)")
print(f"  φ(φ(p)) = φ({phi_p})  = {phi_phi}")
print(f"  Number of primitive roots = φ(φ(p)) = {phi_phi}  ✓")

print(f"\nWhy this matters for Diffie-Hellman:")
print(f"  Using a primitive root as g guarantees the full keyspace.")
print(f"  If g is NOT a primitive root, only a subset of values are reachable,")
print(f"  drastically shrinking the keyspace and making brute-force trivial.")

Q7: PRIMITIVE ROOTS FOR p = 23
Prime p = 23
Definition: g is a primitive root of p if the set
  {g^1 mod p, g^2 mod p, ..., g^(p-1) mod p}
  generates every integer in {1, 2, ..., 22} exactly once.

Testing each candidate g from 2 to p-1:

  g =  2  → missing [5, 7, 10]...  ✗
  g =  3  → missing [5, 7, 10]...  ✗
  g =  4  → missing [5, 7, 10]...  ✗
  g =  5  → powers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]  ✓ PRIMITIVE ROOT
  g =  6  → missing [5, 7, 10]...  ✗
  g =  7  → powers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]  ✓ PRIMITIVE ROOT
  g =  8  → missing [5, 7, 10]...  ✗
  g =  9  → missing [5, 7, 10]...  ✗
  g = 10  → powers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]  ✓ PRIMITIVE ROOT
  g = 11  → powers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]  ✓ PRIMITIVE ROOT
  g = 12  → missing [5, 7, 10]...  ✗
  g = 13  → missing [5, 7

In [8]:
# ============================================================
# Q8: Diffie-Hellman Man-in-the-Middle (MITM) Proxy
# ============================================================

print("=" * 60)
print("Q8: DIFFIE-HELLMAN MITM ATTACK SIMULATION")
print("=" * 60)

p   = 23   # Public prime
g   = 5    # Public generator (primitive root)
a   = 4    # Alice's private secret
b   = 3    # Bob's private secret
eve = 6    # Eve's private secret

print(f"Public parameters:  p = {p},  g = {g}")
print(f"Alice's secret:     a = {a}")
print(f"Bob's secret:       b = {b}")
print(f"Eve's secret:       e = {eve}\n")

# --- Legitimate exchange (no MITM) ---
A_real = pow(g, a, p)
B_real = pow(g, b, p)
print("=" * 40)
print("LEGITIMATE EXCHANGE (without Eve)")
print("=" * 40)
print(f"  Alice sends:  A  = g^a mod p  = {g}^{a} mod {p}  = {A_real}")
print(f"  Bob sends:    B  = g^b mod p  = {g}^{b} mod {p} = {B_real}\n")

# --- Eve's MITM attack ---
forged = pow(g, eve, p)
print("=" * 40)
print("EVE'S MITM ATTACK")
print("=" * 40)
print(f"  Eve intercepts A = {A_real} from Alice and B = {B_real} from Bob.")
print(f"  Eve computes forged value:  g^e mod p = {g}^{eve} mod {p} = {forged}")
print(f"  Eve sends {forged} to Alice  (pretending to be Bob)")
print(f"  Eve sends {forged} to Bob    (pretending to be Alice)\n")

# --- Two separate shared keys ---
K_AE = pow(A_real, eve, p)
K_EB = pow(B_real, eve, p)
print("=" * 40)
print("SHARED KEYS ESTABLISHED BY EVE")
print("=" * 40)
print(f"  K_AE = A^e mod p  = {A_real}^{eve} mod {p}  = {K_AE}")
print(f"         (key shared between Alice and Eve)")
print(f"  K_EB = B^e mod p  = {B_real}^{eve} mod {p} = {K_EB}")
print(f"         (key shared between Bob and Eve)\n")

# --- Verification ---
K_alice_computed = pow(forged, a, p)
K_bob_computed   = pow(forged, b, p)
print("=" * 40)
print("VERIFICATION")
print("=" * 40)
print(f"  Alice computes: (g^e)^a mod p = {forged}^{a} mod {p} = {K_alice_computed}")
print(f"  Matches K_AE ({K_AE})? {K_alice_computed == K_AE}  ✓")
print(f"  Bob computes:   (g^e)^b mod p = {forged}^{b} mod {p} = {K_bob_computed}")
print(f"  Matches K_EB ({K_EB})? {K_bob_computed == K_EB}  ✓\n")

print("RESULT:")
print(f"  Alice believes she shares key {K_AE} with Bob — actually shares it with Eve.")
print(f"  Bob believes he shares key {K_EB} with Alice — actually shares it with Eve.")
print(f"  Eve decrypts Alice's traffic with K_AE, reads/modifies it,")
print(f"  then re-encrypts with K_EB before forwarding to Bob.")
print(f"  Neither Alice nor Bob detects the interception.")

Q8: DIFFIE-HELLMAN MITM ATTACK SIMULATION
Public parameters:  p = 23,  g = 5
Alice's secret:     a = 4
Bob's secret:       b = 3
Eve's secret:       e = 6

LEGITIMATE EXCHANGE (without Eve)
  Alice sends:  A  = g^a mod p  = 5^4 mod 23  = 4
  Bob sends:    B  = g^b mod p  = 5^3 mod 23 = 10

EVE'S MITM ATTACK
  Eve intercepts A = 4 from Alice and B = 10 from Bob.
  Eve computes forged value:  g^e mod p = 5^6 mod 23 = 8
  Eve sends 8 to Alice  (pretending to be Bob)
  Eve sends 8 to Bob    (pretending to be Alice)

SHARED KEYS ESTABLISHED BY EVE
  K_AE = A^e mod p  = 4^6 mod 23  = 2
         (key shared between Alice and Eve)
  K_EB = B^e mod p  = 10^6 mod 23 = 6
         (key shared between Bob and Eve)

VERIFICATION
  Alice computes: (g^e)^a mod p = 8^4 mod 23 = 2
  Matches K_AE (2)? True  ✓
  Bob computes:   (g^e)^b mod p = 8^3 mod 23 = 6
  Matches K_EB (6)? True  ✓

RESULT:
  Alice believes she shares key 2 with Bob — actually shares it with Eve.
  Bob believes he shares key 6 with Al

In [9]:
# ============================================================
# Q9: Proof-of-Work (PoW) Mining Implementation
# ============================================================

import hashlib

print("=" * 60)
print("Q9: PROOF-OF-WORK MINING")
print("=" * 60)

header          = "CSCI6000 Block Data"
target_prefix   = "0000"
nonce           = 0

print(f"Block header:     '{header}'")
print(f"Difficulty target: SHA-256 hash must begin with '{target_prefix}'")
print(f"Starting nonce:    0")
print("\nMining... (searching for valid nonce)\n")

attempts = 0
while True:
    data        = f"{header}{nonce}"
    hash_result = hashlib.sha256(data.encode()).hexdigest()
    attempts   += 1

    if hash_result.startswith(target_prefix):
        break

    nonce += 1

print(f"  ✅ Valid block found after {attempts:,} attempts!")
print(f"\n  Nonce:      {nonce}")
print(f"  Input:      '{header}{nonce}'")
print(f"  SHA-256:    {hash_result}")
print(f"  Prefix:     {hash_result[:4]}  ✓\n")

print("How Proof-of-Work functions:")
print("  The miner repeatedly hashes (Block Header + Nonce) using SHA-256.")
print("  The nonce is incremented by 1 on each failed attempt.")
print("  A valid block is found when the hash starts with the required zeros.")
print("  The difficulty target controls how hard mining is:")
print("    4 leading zeros  → ~1 in 65,536 hashes succeed on average")
print("    In real Bitcoin, target adjusts every 2016 blocks to keep")
print("    average block time at ~10 minutes regardless of total mining power.")

Q9: PROOF-OF-WORK MINING
Block header:     'CSCI6000 Block Data'
Difficulty target: SHA-256 hash must begin with '0000'
Starting nonce:    0

Mining... (searching for valid nonce)

  ✅ Valid block found after 15,961 attempts!

  Nonce:      15960
  Input:      'CSCI6000 Block Data15960'
  SHA-256:    0000ecb73ed803a041833cef0494f9a17ec97b886e5ae6629a53952f25549ccd
  Prefix:     0000  ✓

How Proof-of-Work functions:
  The miner repeatedly hashes (Block Header + Nonce) using SHA-256.
  The nonce is incremented by 1 on each failed attempt.
  A valid block is found when the hash starts with the required zeros.
  The difficulty target controls how hard mining is:
    4 leading zeros  → ~1 in 65,536 hashes succeed on average
    In real Bitcoin, target adjusts every 2016 blocks to keep
    average block time at ~10 minutes regardless of total mining power.


In [10]:
# ============================================================
# Q10: BB84 Quantum Key Distribution Simulation
# ============================================================

import random

print("=" * 60)
print("Q10: BB84 QUANTUM KEY DISTRIBUTION SIMULATION")
print("=" * 60)

random.seed(42)   # Fixed seed — remove this line for a new random run each time
N = 20

# --- Alice generates random bits and random bases ---
alice_bits  = [random.randint(0, 1) for _ in range(N)]
alice_bases = [random.choice(['+', 'x']) for _ in range(N)]

# --- Bob independently chooses random measurement bases ---
bob_bases   = [random.choice(['+', 'x']) for _ in range(N)]

print(f"Simulating BB84 protocol for N = {N} photons\n")
print(f"Bases:  + = Rectilinear (horizontal/vertical)")
print(f"        x = Diagonal (45°/135°)\n")

print(f"Alice's random bits:   {alice_bits}")
print(f"Alice's random bases:  {alice_bases}")
print(f"Bob's random bases:    {bob_bases}\n")

# --- Sifting: keep bits where Alice and Bob chose the same basis ---
sifted_key      = []
sifted_positions = []
match_display   = []

for i in range(N):
    if alice_bases[i] == bob_bases[i]:
        sifted_key.append(alice_bits[i])
        sifted_positions.append(i)
        match_display.append('✓')
    else:
        match_display.append('✗')

print("Basis comparison (position by position):")
print(f"  Match?:  {match_display}\n")

print("Sifting — keeping only bits where bases matched:")
print(f"  Matching positions: {sifted_positions}")
print(f"  Bits retained:      {len(sifted_key)} out of {N} "
      f"({len(sifted_key)/N*100:.1f}%)\n")

sifted_str = ''.join(str(b) for b in sifted_key)
print(f"  ✅ Final Sifted Key: {sifted_key}")
print(f"  As binary string:   {sifted_str}\n")

print("Why ~50% of bits are retained:")
print("  Alice and Bob each choose their basis independently and randomly.")
print("  On average, they agree on 50% of positions.")
print("  Bits measured in the wrong basis give a random result and are discarded.")
print("  The remaining sifted bits are identical for Alice and Bob and form the key.\n")

print("Eavesdropping detection:")
print("  If Eve intercepts photons, she guesses the wrong basis ~50% of the time.")
print("  This disturbs the photon state (Heisenberg Uncertainty Principle).")
print("  Alice and Bob can detect Eve by comparing a sample of sifted bits publicly.")
print("  A high error rate (>11%) in the sample reveals eavesdropping.")

Q10: BB84 QUANTUM KEY DISTRIBUTION SIMULATION
Simulating BB84 protocol for N = 20 photons

Bases:  + = Rectilinear (horizontal/vertical)
        x = Diagonal (45°/135°)

Alice's random bits:   [0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1]
Alice's random bases:  ['+', '+', 'x', 'x', 'x', '+', '+', 'x', '+', '+', 'x', '+', 'x', 'x', 'x', '+', 'x', '+', 'x', '+']
Bob's random bases:    ['x', 'x', '+', '+', '+', '+', 'x', '+', '+', '+', 'x', 'x', 'x', 'x', '+', 'x', 'x', '+', 'x', '+']

Basis comparison (position by position):
  Match?:  ['✗', '✗', '✗', '✗', '✗', '✓', '✗', '✗', '✓', '✓', '✓', '✗', '✓', '✓', '✗', '✗', '✓', '✓', '✓', '✓']

Sifting — keeping only bits where bases matched:
  Matching positions: [5, 8, 9, 10, 12, 13, 16, 17, 18, 19]
  Bits retained:      10 out of 20 (50.0%)

  ✅ Final Sifted Key: [0, 1, 0, 0, 0, 0, 1, 0, 1, 1]
  As binary string:   0100001011

Why ~50% of bits are retained:
  Alice and Bob each choose their basis independently and randomly.
  On